In [ ]:
# Install necessary dependencies.
using Pkg
Pkg.activate(; temp=true)
Pkg.add([])

```julia
#| echo: false
#| output: false
using Pkg;
Pkg.instantiate();
```

Bayesian logistic regression is the Bayesian counterpart to a common tool in machine learning, logistic regression.
The goal of logistic regression is to predict a one or a zero for a given training item.
An example might be predicting whether someone is sick or ill given their symptoms and personal information.

In our example, we'll be working to predict whether someone is likely to default with a synthetic dataset found in the `RDatasets` package. This dataset, `Defaults`, comes from R's [ISLR](https://cran.r-project.org/web/packages/ISLR/index.html) package and contains information on borrowers.

To start, let's import all the libraries we'll need.

```julia
using Turing
using RDatasets
using Plots, StatsPlots
using StatsFuns: logistic
using MLUtils: splitobs
using StatsBase: fit, transform!, ZScoreTransform

# Set a seed for reproducibility.
using Random
Random.seed!(0);
```

## Data Cleaning & Set Up

Now we're going to import our dataset.
The first six rows of the dataset are shown below so you can get a good feel for what kind of data we have.

```julia
# Import the "Default" dataset.
data = RDatasets.dataset("ISLR", "Default");

# Show the first six rows of the dataset.
first(data, 6)
```

Most machine learning processes require some effort to tidy up the data, and this is no different.
We need to convert the `Default` and `Student` columns, which say "Yes" or "No" into 1s and 0s.
Afterwards, we'll get rid of the old words-based columns.

```julia
# Convert "Default" and "Student" to numeric values.
yesno = ["No", "Yes"]
data[!, :DefaultNum] = indexin(data[!, :Default], yesno) .- 1
data[!, :StudentNum] = indexin(data[!, :Student], yesno) .- 1

# Delete the old columns which say "Yes" and "No".
select!(data, Not([:Default, :Student]))

# Show the first six rows of our edited dataset.
first(data, 6)
```

Our predictor variables are `StudentNum`, `Balance`, and `Income`, and our target variable is `DefaultNum`; we separate those out into `X` and `Y` for ease of use later on.
We'll also convert them to `Matrix` and `Vector` types, respectively, by wrapping them in `Array`.

```julia
# `splitobs` later expects that `X` has observations in columns,
# hence the transpose on `X`.
X = Array(data[!, [:StudentNum, :Balance, :Income]])'
Y = Array(data[!, :DefaultNum])
size(X), size(Y)
```

It's now time to split our dataset into training and testing sets.
We separate our data into two partitions, `train` and `test`.
You can use a higher percentage of splitting (or a lower one) by modifying the `at = 0.05` argument.
Here, we are training on only 5% of the data in order to highlight the power of Bayesian inference with small sample sizes.
To do the splitting we will leverage `MLUtils`, which also lets us effortlessly shuffle our observations and perform a [stratified split](https://en.wikipedia.org/wiki/Stratified_sampling) to get a representative test set.

```julia
(train_X, train_Y), (test_X, test_Y) = splitobs((X, Y); at=0.05, shuffle=true, stratified=Y)

# Let's check that the labels are distributed
# similarly in the training and test sets.
mean(train_Y), mean(test_Y)
```

We must now rescale our numeric variables so that they are centred around zero by subtracting each column by the mean and dividing it by the standard deviation.
This rescaling ensures features are on comparable scales, which improves sampler initialisation and convergence.

Note here that we leave out the `StudentNum` variable (row 1) from the normalisation, since it is already binary and doesn't need to be rescaled.

In [ ]:
dt = fit(ZScoreTransform, view(train_X, 2:3, :); dims=2)
transform!(dt, view(train_X, 2:3, :))
transform!(dt, view(test_X, 2:3, :))

## Model Declaration

Finally, we can define our model.

`logistic_regression` takes four arguments:

  - `x` is our set of independent variables;
  - `y` is the element we want to predict;
  - `σ` is the (fixed) standard deviation we want to assume for our priors.

Within the model, we create four coefficients (`intercept`, `student`, `balance`, and `income`) and assign a prior of normally distributed with means of zero and standard deviations of `σ`.
We want to find values of these four coefficients to predict any given `y`.

The `for` block creates a variable `v` which is the logistic function. We then observe the likelihood of calculating `v` given the actual label, `y[i]`.

In [ ]:
@model function logistic_regression(x, y, σ)
    N = size(x, 2)
    @assert length(y) == N

    intercept ~ Normal(0, σ)
    student ~ Normal(0, σ)
    balance ~ Normal(0, σ)
    income ~ Normal(0, σ)

    for i in 1:N
        v = logistic(intercept + student * x[1, i] + balance * x[2, i] + income * x[3, i])
        y[i] ~ Bernoulli(v)
    end
end;

## Sampling

Now we can run our sampler.
Here we'll use [`NUTS`](https://turinglang.org/Turing.jl/stable/api/Inference/#Turing.Inference.NUTS) to sample from our posterior.

```julia
#| output: false
setprogress!(false)
```

```julia
# Sample using NUTS.
m = logistic_regression(train_X, train_Y, 1.0)
chain = sample(m, NUTS(), MCMCThreads(), 1_500, 3)
```

> ## Sampling With Multiple Threads
> The `sample()` call above assumes that you have at least `nchains` threads available in your Julia instance.
> If you do not, the multiple chains will run sequentially, and you may notice a warning.
> For more information, see [the Turing documentation on sampling multiple chains]({{<meta core-functionality>}}#sampling-multiple-chains).

```julia
#| echo: false
let
    mean_params = mean(chain)
    @assert mean_params[@varname(student)] < 0.1
    @assert mean_params[@varname(balance)] > 1
end
```

Since we ran multiple chains, we may as well do a spot check to make sure each chain converges around similar points.

In [ ]:
plot(chain)

```julia
#| echo: false
using FlexiChains
let
    mean_per_chain = mean(chain; dims=:iter)
    for parameter in FlexiChains.parameters(mean_per_chain)
        @assert isapprox(mean_per_chain[parameter, chain=2], mean_per_chain[parameter, chain=1]; rtol=0.1)
        @assert isapprox(mean_per_chain[parameter, chain=3], mean_per_chain[parameter, chain=1]; rtol=0.1)
    end
end
```

Looks good!

We can also use the `cornerplot` function from StatsPlots to show the distributions of the various parameters of our logistic regression.

In [ ]:
StatsPlots.cornerplot(chain, [@varname(student), @varname(balance), @varname(income)])

Fortunately the corner plot appears to demonstrate unimodal distributions for each of our parameters, so it should be straightforward to take the means of each parameter's sampled values to estimate our model to make predictions.

## Making Predictions

How do we test how well the model actually predicts whether someone is likely to default?
We need to build a prediction function that takes the `test` object we made earlier and runs it through the average parameter calculated during sampling.

The `prediction` function below takes a `Matrix` and a `Chain` object.
It takes the mean of each parameter's sampled values and re-runs the logistic function using those mean values for every element in the test set.

```julia
function prediction(x::AbstractMatrix, chain, threshold)
    # Pull the means from each parameter's sampled values in the chain.
    intercept = mean(chain[@varname(intercept)])
    student = mean(chain[@varname(student)])
    balance = mean(chain[@varname(balance)])
    income = mean(chain[@varname(income)])

    # Retrieve the number of observations.
    n = size(x, 2)

    # Generate a vector to store our predictions.
    v = Vector{Bool}(undef, n)

    # Calculate the logistic function for each element in the test set.
    for i in 1:n
        num = logistic(
            intercept .+ student * x[1, i] + balance * x[2, i] + income * x[3, i]
        )
        v[i] = num >= threshold
    end
    return v
end
```

Let's see how we did!
We run the test matrix through the prediction function, and compute the [mean squared error](https://en.wikipedia.org/wiki/Mean_squared_error) (MSE) for our prediction.
The `threshold` variable sets the decision boundary for classification.
For example, a threshold of 0.07 will predict a default (value of 1) for any predicted probability greater than 0.07 and no default otherwise.
Lower thresholds increase sensitivity but may increase false positives.

```julia
# Set the prediction threshold.
threshold = 0.07

# Make the predictions.
predictions = prediction(test_X, chain, threshold)

# Calculate MSE for our test set.
loss = sum((predictions - test_Y) .^ 2) / length(test_Y)
```

Perhaps more important is to see what percentage of defaults we correctly predicted.
The code below simply counts defaults and predictions and presents the results.

In [ ]:
defaults = sum(test_Y)
not_defaults = length(test_Y) - defaults

predicted_defaults = sum(test_Y .== predictions .== 1)
predicted_not_defaults = sum(test_Y .== predictions .== 0)

println("Defaults: $defaults
    Predictions: $predicted_defaults
    Percentage defaults correct $(predicted_defaults/defaults)")

println("Not defaults: $not_defaults
    Predictions: $predicted_not_defaults
    Percentage non-defaults correct $(predicted_not_defaults/not_defaults)")

```julia
#| echo: false
let
    percentage_correct = predicted_defaults / defaults
    @assert 0.6 < percentage_correct
end
```

The above shows that with a threshold of 0.07, we correctly predict a respectable portion of the defaults, and correctly identify most non-defaults.
This is fairly sensitive to a choice of threshold, and you may wish to experiment with it.